# FAISS Vector Retrieval

## Goal

Build a production-style vector retrieval engine over product embeddings.

## Why

In Notebook 05 we generated semantic embeddings for all products.

To find similar products, we used cosine similarity against the entire catalog.

This approach becomes inefficient as catalog size grows.

FAISS (Facebook AI Similarity Search) enables fast nearest-neighbor retrieval over large embedding collections.

## Output

- FAISS index
- Millisecond product retrieval
- Semantic search infrastructure for the hybrid recommender

In [2]:
import pandas as pd
import numpy as np
import faiss

In [ ]:
catalog = pd.read_parquet(
    "../data/catalog.parquet"
)

embeddings = np.load(
    "../data/product_embeddings.npy"
)

print(catalog.shape)
print(embeddings.shape)

print(embeddings.dtype)
print(embeddings.shape)

(63001, 6)
(63001, 384)
float32
(63001, 384)


# Build Vector Index

Create a FAISS index over product embeddings.

The index stores semantic representations of products and supports efficient nearest-neighbor retrieval.

In [12]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(
    dimension
)

index.add(embeddings)

print(
    f"Vectors Indexed: {index.ntotal:,}"
)

faiss.write_index(
    index,
    "../data/faiss_index.bin"
)

Vectors Indexed: 63,001


In [13]:
def faiss_similar_products(
    asin,
    top_k=10
):

    matches = catalog.index[
        catalog["asin"] == asin
    ]

    if len(matches) == 0:
        return None

    idx = matches[0]

    query_vector = embeddings[
        idx
    ].reshape(1, -1)

    scores, indices = index.search(
        query_vector,
        top_k + 1
    )

    result_idx = indices[0][1:]

    return catalog.iloc[
        result_idx
    ][
        [
            "asin",
            "title"
        ]
    ]

In [14]:
sample_asin = catalog.iloc[0]["asin"]

faiss_similar_products(
    sample_asin,
    top_k=10
)

,asin,title
37525,B004P7GAPI,Rand McNally Intelliroute TND 710 Truck GPS
37655,B004Q3R91K,Rand McNally Intelliroute TND 510 Truck GPS
58131,B00C7FKT2A,Rand McNally Intelliroute TND 520 Truck GPS wi...
46402,B006ZOI9OY,Rand McNally TND 720 LM IntelliRoute Truck GPS...
29691,B003HBC9TY,Garmin n&uuml;vi 1450T 5-Inch Portable GPS Nav...
56604,B00B2F4O86,Rand McNally Foris 850 Outdoor GPS
43397,B005OEIY8W,Rand McNally TripMaker RVND 7710 7-Inch GPS fo...
20973,B001TH76JQ,Garmin nuvi 465/465T 4.3-Inch Widescreen Bluet...
39325,B004YJYQPS,Rand McNally TripMaker RVND 5510 5-Inch RV GPS
15631,B0014LC9S0,Magellan RoadMate 1412 4.3-Inch Portable GPS N...


In [16]:
import time
sample_asin = catalog.iloc[
    100
]["asin"]

start = time.time()

faiss_similar_products(
    sample_asin,
    top_k=10
)

end = time.time()

print(
    f"Latency: {(end-start)*1000:.2f} ms"
)

Latency: 17.81 ms


# Conclusions

FAISS successfully indexed 63,001 product embeddings and enabled efficient semantic retrieval.

Key observations:

- Retrieval quality matched the brute-force embedding search from Notebook 05.
- Query latency was reduced to milliseconds.
- The FAISS index provides scalable infrastructure for future hybrid recommendation systems.

This component will serve as the content-based retrieval layer in the final hybrid recommender architecture.